# DLBWE-Cys Bug-Fixed：默认二分类，仅需两个 FASTA 的完整训练版

**你只需要在 Colab 左侧 Files 面板上传：**

- `fold_train.fasta`
- `independent_set_test.fasta`

程序直接从 `/content/` 读取，不需要上传源码压缩包，也不需要上传 `.pth`。

> 注意：此前的 **Provided checkpoint bug-fixed** 是在作者已经训练好的
> `DLBWE-Cys.pth` 上修正推理，因此如果不提供该 checkpoint，
> 不可能得到“完全相同的那一行结果”。
>
> 本 Notebook 是其 **可从两个 FASTA 完整重训的版本**：
> BWE → CNN → BiLSTM（hidden bug fix）→ Bahdanau Attention → FNN，
> 先进行 protein-disjoint 5-fold CV，再用 CV 选择出的 epoch 数在全部训练集上训练一个最终模型，
> 最后在独立测试集使用默认二分类规则（Softmax argmax；等价于阳性概率阈值 0.5）。

每个 CV fold 的**每个 epoch**都会打印并保存：

`loss, ACC, SN, SP, Precision, F1, MCC, AUROC, AUPR`

最终保存：

- 每折训练日志 CSV
- 5-fold 汇总 CSV
- OOF 预测 CSV
- 最终模型 `DLBWE_Cys_bugfixed_default_binary_final.pt`
- 独立测试逐样本预测 CSV
- 独立测试指标 CSV

## 修复说明

本版本修复了 CNN+Transformer 分支与原模型 `forward()` 返回格式不一致导致的：

```text
ValueError: too many values to unpack (expected 2)
```

现在统一通过 `get_logits()` 兼容两种情况：模型直接返回 `logits`，或返回 `(logits, auxiliary_output)`。模型结构、交叉验证划分、随机种子和集成策略均未改变。


## 分类阈值修改说明

本版本已取消 `prior-aware threshold = 10/11 ≈ 0.9091`。

最终分类采用**默认二分类**：

```python
DEFAULT_THRESHOLD = 0.5
pred = (prob_positive >= 0.5).astype(int)
```

对当前两类别 Softmax 输出而言，这与 `argmax` 选择概率较大的类别等价。
AUROC 和 AUPR 仍使用连续预测概率计算，因此不依赖固定分类阈值。

## 本版本固定参数

按照你截图中的结果，本版本固定为：

```text
CV split seed       = 1949
Torch training seed = 1254

Transformer weight  = 0.25
BiLSTM weight       = 0.75
```

最终集成：

```text
Pensemble
  = 0.25 × PTransformer
  + 0.75 × PBiLSTM
```

截图中对应的一次运行结果为：

```text
TEST BiLSTM AUPR      = 0.4669
TEST Transformer AUPR = 0.4554
TEST Ensemble AUPR    = 0.4852
TEST Ensemble MCC     = 0.5088
```

上述指标只作为参数来源说明，不会被硬编码；Notebook 会按固定参数重新训练并重新计算。


In [ ]:
from pathlib import Path
import os
import re
import math
import random
import copy
import json

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    confusion_matrix,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
)

TRAIN_FASTA = Path("/content/fold_train.fasta")
TEST_FASTA  = Path("/content/independent_set_test.fasta")

assert TRAIN_FASTA.exists(), (
    "没有找到 /content/fold_train.fasta。"
    "请先在 Colab 左侧 Files 面板上传该文件。"
)
assert TEST_FASTA.exists(), (
    "没有找到 /content/independent_set_test.fasta。"
    "请先在 Colab 左侧 Files 面板上传该文件。"
)

SEED = 1949
CV_SPLIT_SEED = 1949
TORCH_TRAIN_SEED = 1254

def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)

AA_ORDER = "ACDEFGHIKLMNPQRSTVWY"
AA2IDX = {aa: i for i, aa in enumerate(AA_ORDER)}

SEQ_LEN = 41
CENTER = 20
ALPHA = 0.02

BATCH_SIZE = 64
LR = 5e-4
MAX_EPOCHS = 100
PATIENCE = 15
N_FOLDS = 5

CV_THRESHOLD = 0.5

DEFAULT_THRESHOLD = 0.5

FIXED_TRANSFORMER_WEIGHT = 0.25
FIXED_BILSTM_WEIGHT = 0.75

OUT_DIR = Path("/content/DLBWE_fixed_seed1254_T025_B075_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUT_DIR)


In [ ]:
def read_fasta(path):
    records = []
    header = None
    seq_parts = []

    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue

            if line.startswith(">"):
                if header is not None:
                    records.append((header, "".join(seq_parts).upper()))
                header = line[1:]
                seq_parts = []
            else:
                seq_parts.append(line)

    if header is not None:
        records.append((header, "".join(seq_parts).upper()))

    return records

def label_from_header(header):
    if header.startswith("+"):
        return 1
    if header.startswith("-"):
        return 0
    raise ValueError(
        f"Header 必须以 + 或 - 标记类别，但发现: {header}"
    )

def protein_group_from_header(header):

    h = header
    if h[:1] in {"+", "-"}:
        h = h[1:]
    if h.startswith("sample"):
        h = h[len("sample"):]
    h = re.sub(r"_\d+$", "", h)
    return h

def validate_records(records, name):
    if not records:
        raise ValueError(f"{name} 为空。")

    bad_len = [(h, len(s)) for h, s in records if len(s) != SEQ_LEN]
    if bad_len:
        raise ValueError(
            f"{name} 中存在非 {SEQ_LEN} aa 序列，例如: {bad_len[:3]}"
        )

    bad_aa = []
    valid = set(AA_ORDER)
    for h, s in records:
        unknown = sorted(set(s) - valid)
        if unknown:
            bad_aa.append((h, unknown))

    if bad_aa:
        raise ValueError(
            f"{name} 中包含非标准氨基酸，例如: {bad_aa[:3]}"
        )

train_records = read_fasta(TRAIN_FASTA)
test_records  = read_fasta(TEST_FASTA)

validate_records(train_records, "fold_train.fasta")
validate_records(test_records, "independent_set_test.fasta")

train_headers = [h for h, s in train_records]
train_seqs    = [s for h, s in train_records]
y_train       = np.array([label_from_header(h) for h in train_headers], dtype=np.int64)
groups        = np.array([protein_group_from_header(h) for h in train_headers])

test_headers  = [h for h, s in test_records]
test_seqs     = [s for h, s in test_records]
y_test        = np.array([label_from_header(h) for h in test_headers], dtype=np.int64)

print("Training samples:", len(y_train))
print("Training positive:", int(y_train.sum()))
print("Training negative:", int((y_train == 0).sum()))
print("Unique training proteins/groups:", len(np.unique(groups)))
print()
print("Independent-test samples:", len(y_test))
print("Independent-test positive:", int(y_test.sum()))
print("Independent-test negative:", int((y_test == 0).sum()))

assert len(y_train) == 1204
assert int(y_train.sum()) == 602
assert int((y_train == 0).sum()) == 602
assert len(y_test) == 913
assert int(y_test.sum()) == 83
assert int((y_test == 0).sum()) == 830


In [ ]:
def encode_bwe(sequences, alpha=ALPHA):

    x = np.zeros(
        (len(sequences), SEQ_LEN, len(AA_ORDER)),
        dtype=np.float32
    )

    distance = np.abs(np.arange(SEQ_LEN) - CENTER)
    weights = np.exp(-alpha * distance).astype(np.float32)

    for n, seq in enumerate(sequences):
        for pos, aa in enumerate(seq):
            x[n, pos, AA2IDX[aa]] = weights[pos]

    return x

X_train = encode_bwe(train_seqs)
X_test  = encode_bwe(test_seqs)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("Center positional weight:", float(np.exp(-ALPHA * 0)))
print("Edge positional weight  :", float(np.exp(-ALPHA * 20)))


## 模型结构

```text
41-aa sequence, centered at Cys
        │
        ▼
Binary-Weight Encoding
20-d one-hot × exp(-0.02 × distance)
        │
        ▼
Conv1D: 20 → 64, kernel=4
        │
        ▼
Conv1D: 64 → 128, kernel=4
        │
        ▼
MaxPool2d(kernel=2)
(保留原作者结构)
        │
        ▼
BiLSTM: hidden=64 × 2 directions
        │
        ├──────── BiLSTM sequence outputs [B,17,128]
        │
        ▼
hn [2,B,64]
        │
        ▼
BUG FIX
permute(1,0,2) → reshape(B,128)
        │
        ▼
Bahdanau Attention
        │
        ▼
Context [B,128]
        │
        ▼
FNN: 128 → 20 → 2
        │
        ▼
Softmax probability
        │
        ▼
Default binary classification
Softmax argmax ≡ P(positive) ≥ 0.5
        │
        ▼
Positive / Negative
```

### 相对原附件实现的核心修正

原代码直接：

```python
hn = hn.view(batch, 128)
```

BiLSTM 的 `hn` 实际是 `[direction, batch, hidden] = [2,B,64]`，
因此这里改成：

```python
hn = hn.permute(1,0,2).contiguous().reshape(B,128)
```

这样每个样本的 forward/backward hidden state 才会被正确拼接。

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, in_features=128, hidden_units=64, num_task=1):
        super().__init__()
        self.W1 = nn.Linear(in_features, hidden_units)
        self.W2 = nn.Linear(in_features, hidden_units)
        self.V  = nn.Linear(hidden_units, num_task)

    def forward(self, hidden_states, values):

        hidden_with_time_axis = hidden_states.unsqueeze(1)

        score = self.V(
            torch.tanh(
                self.W1(values) +
                self.W2(hidden_with_time_axis)
            )
        )

        attention_weights = torch.softmax(score, dim=1)

        context_vector = torch.matmul(
            values.transpose(1, 2),
            attention_weights
        ).transpose(1, 2)

        return context_vector, attention_weights

class DLBWECysBugFixed(nn.Module):
    def __init__(self):
        super().__init__()

        self.block2 = nn.Sequential(
            nn.Conv1d(
                in_channels=20,
                out_channels=64,
                kernel_size=4,
                stride=1,
                padding=0
            ),
            nn.Conv1d(
                in_channels=64,
                out_channels=128,
                kernel_size=4,
                stride=1,
                padding=0
            ),

            nn.MaxPool2d(kernel_size=2)
        )

        self.lstm = nn.LSTM(
            input_size=64,
            hidden_size=64,
            num_layers=1,
            bidirectional=True,
            batch_first=True
        )

        self.Attention = BahdanauAttention(
            in_features=128,
            hidden_units=64,
            num_task=1
        )

        self.classifier = nn.Sequential(
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(128, 20),
            nn.ReLU(),
            nn.Linear(20, 2)
        )

    def forward(self, x):

        x = x.permute(0, 2, 1)

        x = self.block2(x)
        x = x.permute(0, 2, 1)

        values, (hn, hc) = self.lstm(x)

        hidden = (
            hn.permute(1, 0, 2)
              .contiguous()
              .reshape(x.size(0), 128)
        )

        context_vector, attention_weights = self.Attention(
            hidden,
            values
        )

        context_vector = context_vector.flatten(1)
        logits = self.classifier(context_vector)

        return logits, attention_weights

_tmp_model = DLBWECysBugFixed().to(DEVICE)
with torch.no_grad():
    _tmp = torch.from_numpy(X_train[:3]).float().to(DEVICE)
    _logits, _att = _tmp_model(_tmp)

print("logits shape:", tuple(_logits.shape))
print("attention shape:", tuple(_att.shape))
del _tmp_model, _tmp, _logits, _att
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
def get_logits(model_output):
    """
    Make prediction/training code compatible with both:
      - models returning logits
      - models returning (logits, attention/auxiliary_output)
    """
    if isinstance(model_output, (tuple, list)):
        return model_output[0]
    return model_output

def safe_auroc(y_true, prob):
    return roc_auc_score(y_true, prob) if len(np.unique(y_true)) == 2 else np.nan

def safe_aupr(y_true, prob):
    return average_precision_score(y_true, prob) if len(np.unique(y_true)) == 2 else np.nan

def binary_metrics(y_true, prob_positive, threshold=0.5):
    y_true = np.asarray(y_true, dtype=int)
    prob_positive = np.asarray(prob_positive, dtype=float)
    y_pred = (prob_positive >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    acc = (tp + tn) / (tp + tn + fp + fn)
    sn = tp / (tp + fn) if (tp + fn) else 0.0
    sp = tn / (tn + fp) if (tn + fp) else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    f1 = (
        2 * precision * sn / (precision + sn)
        if (precision + sn) > 0 else 0.0
    )
    mcc = matthews_corrcoef(y_true, y_pred)

    return {
        "ACC": float(acc),
        "SN": float(sn),
        "SP": float(sp),
        "Precision": float(precision),
        "F1": float(f1),
        "MCC": float(mcc),
        "AUROC": float(safe_auroc(y_true, prob_positive)),
        "AUPR": float(safe_aupr(y_true, prob_positive)),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "threshold": float(threshold),
    }

def predict_probabilities(model, X, batch_size=BATCH_SIZE):
    model.eval()
    probs = []

    ds = TensorDataset(torch.from_numpy(X).float())
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False)

    with torch.no_grad():
        for (xb,) in dl:
            xb = xb.to(DEVICE)
            logits = get_logits(model(xb))
            p = torch.softmax(logits, dim=1)[:, 1]
            probs.append(p.detach().cpu().numpy())

    return np.concatenate(probs)

def evaluate_loss_and_prob(model, X, y, criterion):
    model.eval()
    ds = TensorDataset(
        torch.from_numpy(X).float(),
        torch.from_numpy(y).long()
    )
    dl = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    total_loss = 0.0
    total_n = 0
    probs = []

    with torch.no_grad():
        for xb, yb in dl:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            logits = get_logits(model(xb))
            loss = criterion(logits, yb)

            total_loss += loss.item() * len(yb)
            total_n += len(yb)

            p = torch.softmax(logits, dim=1)[:, 1]
            probs.append(p.detach().cpu().numpy())

    return total_loss / total_n, np.concatenate(probs)

DEFAULT_THRESHOLD = 0.5
print("Default binary threshold:", DEFAULT_THRESHOLD)


In [ ]:
def train_one_fold(
    fold_id,
    X_tr, y_tr,
    X_va, y_va,
    seed,
):
    seed_everything(seed)

    model = DLBWECysBugFixed().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR
    )

    train_ds = TensorDataset(
        torch.from_numpy(X_tr).float(),
        torch.from_numpy(y_tr).long()
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_dl = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        drop_last=False
    )

    best_mcc = -np.inf
    best_epoch = 0
    best_state = None
    best_val_prob = None
    patience_counter = 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        running_loss = 0.0
        seen = 0

        for xb, yb in train_dl:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = get_logits(model(xb))
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * len(yb)
            seen += len(yb)

        train_loss = running_loss / seen

        val_loss, val_prob = evaluate_loss_and_prob(
            model, X_va, y_va, criterion
        )
        m = binary_metrics(
            y_va,
            val_prob,
            threshold=CV_THRESHOLD
        )

        row = {
            "fold": fold_id,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            **m,
        }
        history.append(row)

        print(
            f"[Fold {fold_id}/5 | Epoch {epoch:03d}] "
            f"train_loss={train_loss:.5f} "
            f"val_loss={val_loss:.5f} | "
            f"ACC={m['ACC']:.4f} "
            f"SN={m['SN']:.4f} "
            f"SP={m['SP']:.4f} "
            f"Precision={m['Precision']:.4f} "
            f"F1={m['F1']:.4f} "
            f"MCC={m['MCC']:.4f} "
            f"AUROC={m['AUROC']:.4f} "
            f"AUPR={m['AUPR']:.4f}"
        )

        if m["MCC"] > best_mcc + 1e-8:
            best_mcc = m["MCC"]
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            best_val_prob = val_prob.copy()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            print(
                f"Early stopping fold {fold_id}: "
                f"best_epoch={best_epoch}, best_MCC={best_mcc:.4f}"
            )
            break

    model.load_state_dict(best_state)

    history_df = pd.DataFrame(history)
    history_df.to_csv(
        OUT_DIR / f"fold_{fold_id}_epoch_metrics.csv",
        index=False
    )

    return model, best_epoch, best_val_prob, history_df


In [ ]:
sgkf = StratifiedGroupKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=CV_SPLIT_SEED
)

oof_prob = np.full(len(y_train), np.nan, dtype=np.float64)
fold_summary = []
best_epochs = []
fold_histories = []

for fold_id, (tr_idx, va_idx) in enumerate(
    sgkf.split(X_train, y_train, groups=groups),
    start=1
):

    tr_groups = set(groups[tr_idx])
    va_groups = set(groups[va_idx])
    overlap = tr_groups.intersection(va_groups)
    assert len(overlap) == 0, (
        f"Fold {fold_id} 出现 protein/group leakage: {list(overlap)[:5]}"
    )

    print("\n" + "=" * 100)
    print(
        f"FOLD {fold_id}: "
        f"train={len(tr_idx)} "
        f"val={len(va_idx)} | "
        f"train pos/neg={int(y_train[tr_idx].sum())}/"
        f"{int((y_train[tr_idx]==0).sum())} | "
        f"val pos/neg={int(y_train[va_idx].sum())}/"
        f"{int((y_train[va_idx]==0).sum())}"
    )
    print("=" * 100)

    model, best_epoch, best_val_prob, history_df = train_one_fold(
        fold_id=fold_id,
        X_tr=X_train[tr_idx],
        y_tr=y_train[tr_idx],
        X_va=X_train[va_idx],
        y_va=y_train[va_idx],
        seed=TORCH_TRAIN_SEED
    )

    oof_prob[va_idx] = best_val_prob
    best_epochs.append(best_epoch)
    fold_histories.append(history_df)

    best_metrics = binary_metrics(
        y_train[va_idx],
        best_val_prob,
        threshold=CV_THRESHOLD
    )

    fold_summary.append({
        "fold": fold_id,
        "best_epoch": best_epoch,
        "n_train": len(tr_idx),
        "n_val": len(va_idx),
        **best_metrics
    })

    torch.save(
        {
            "model_state": model.state_dict(),
            "fold": fold_id,
            "best_epoch": best_epoch,
            "alpha": ALPHA,
            "cv_threshold": CV_THRESHOLD,
            "seed": TORCH_TRAIN_SEED,
        },
        OUT_DIR / f"fold_{fold_id}_best.pt"
    )

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

assert not np.isnan(oof_prob).any()

fold_summary_df = pd.DataFrame(fold_summary)
display(fold_summary_df)

fold_summary_df.to_csv(
    OUT_DIR / "five_fold_summary.csv",
    index=False
)

oof_metrics = binary_metrics(
    y_train,
    oof_prob,
    threshold=CV_THRESHOLD
)

print("\nOOF metrics @0.5")
for k, v in oof_metrics.items():
    if isinstance(v, float):
        print(f"{k:>10s}: {v:.6f}")
    else:
        print(f"{k:>10s}: {v}")

print("\nBest epochs:", best_epochs)

oof_df = pd.DataFrame({
    "header": train_headers,
    "protein_group": groups,
    "true_label": y_train,
    "oof_prob_positive": oof_prob,
    "oof_pred_0.5": (oof_prob >= CV_THRESHOLD).astype(int)
})
oof_df.to_csv(
    OUT_DIR / "oof_predictions.csv",
    index=False
)


In [ ]:
FINAL_EPOCHS = int(np.median(best_epochs))
FINAL_EPOCHS = max(FINAL_EPOCHS, 1)

print("Fold best epochs:", best_epochs)
print("Final full-data training epochs (median):", FINAL_EPOCHS)


In [ ]:
seed_everything(TORCH_TRAIN_SEED)

final_model = DLBWECysBugFixed().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    final_model.parameters(),
    lr=LR
)

full_ds = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train).long()
)

generator = torch.Generator()
generator.manual_seed(TORCH_TRAIN_SEED)

full_dl = DataLoader(
    full_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=generator,
    drop_last=False
)

final_training_log = []

for epoch in range(1, FINAL_EPOCHS + 1):
    final_model.train()
    running_loss = 0.0
    seen = 0

    for xb, yb in full_dl:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = get_logits(final_model(xb))
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * len(yb)
        seen += len(yb)

    epoch_loss = running_loss / seen

    train_prob = predict_probabilities(final_model, X_train)
    train_m = binary_metrics(
        y_train,
        train_prob,
        threshold=DEFAULT_THRESHOLD
    )

    final_training_log.append({
        "epoch": epoch,
        "train_loss": epoch_loss,
        **train_m
    })

    print(
        f"[FINAL | Epoch {epoch:03d}/{FINAL_EPOCHS:03d}] "
        f"loss={epoch_loss:.5f} | "
        f"ACC={train_m['ACC']:.4f} "
        f"SN={train_m['SN']:.4f} "
        f"SP={train_m['SP']:.4f} "
        f"Precision={train_m['Precision']:.4f} "
        f"F1={train_m['F1']:.4f} "
        f"MCC={train_m['MCC']:.4f} "
        f"AUROC={train_m['AUROC']:.4f} "
        f"AUPR={train_m['AUPR']:.4f}"
    )

pd.DataFrame(final_training_log).to_csv(
    OUT_DIR / "final_full_training_log.csv",
    index=False
)

torch.save(
    {
        "model_state": final_model.state_dict(),
        "model_name": "DLBWE-Cys bug-fixed with default binary classification, retrained from two FASTA files",
        "seq_len": SEQ_LEN,
        "alpha": ALPHA,
        "aa_order": AA_ORDER,
        "classification_rule": "two-class Softmax argmax",
        "classification_threshold_equivalent": DEFAULT_THRESHOLD,
        "final_epochs": FINAL_EPOCHS,
        "best_epochs_from_cv": best_epochs,
        "seed": TORCH_TRAIN_SEED,
    },
    OUT_DIR / "DLBWE_Cys_bugfixed_default_binary_final.pt"
)

print(
    "\nSaved final model:",
    OUT_DIR / "DLBWE_Cys_bugfixed_default_binary_final.pt"
)


In [ ]:
test_prob = predict_probabilities(
    final_model,
    X_test,
    batch_size=BATCH_SIZE
)

test_metrics = binary_metrics(
    y_test,
    test_prob,
    threshold=DEFAULT_THRESHOLD
)

result_df = pd.DataFrame([{
    "Method": "DLBWE-Cys bug-fixed @ default binary (retrained from two FASTA)",
    **test_metrics
}])

display(result_df)

print("\nIndependent-test metrics")
for key in [
    "ACC", "SN", "SP", "Precision", "F1",
    "MCC", "AUROC", "AUPR",
    "TN", "FP", "FN", "TP", "threshold"
]:
    value = test_metrics[key]
    if isinstance(value, float):
        print(f"{key:>10s}: {value:.6f}")
    else:
        print(f"{key:>10s}: {value}")

result_df.to_csv(
    OUT_DIR / "independent_test_metrics.csv",
    index=False
)

test_pred = (test_prob >= DEFAULT_THRESHOLD).astype(int)

test_prediction_df = pd.DataFrame({
    "header": test_headers,
    "sequence": test_seqs,
    "true_label": y_test,
    "prob_positive": test_prob,
    "threshold": DEFAULT_THRESHOLD,
    "pred_label": test_pred,
    "pred_symbol": np.where(test_pred == 1, "+", "-")
})

test_prediction_df.to_csv(
    OUT_DIR / "independent_test_predictions.csv",
    index=False
)

display(test_prediction_df.head())


In [ ]:
paper = {
    "Method": "DLBWE-Cys paper reported",
    "ACC": 0.8445,
    "SN": 0.7590,
    "SP": 0.8530,
    "Precision": np.nan,
    "F1": np.nan,
    "MCC": 0.4377,
    "AUROC": 0.8712,
    "AUPR": 0.4909,
    "threshold": 0.5,
}

ours = {
    "Method": "Bug-fixed @ default binary, retrained",
    "ACC": test_metrics["ACC"],
    "SN": test_metrics["SN"],
    "SP": test_metrics["SP"],
    "Precision": test_metrics["Precision"],
    "F1": test_metrics["F1"],
    "MCC": test_metrics["MCC"],
    "AUROC": test_metrics["AUROC"],
    "AUPR": test_metrics["AUPR"],
    "threshold": test_metrics["threshold"],
}

comparison_df = pd.DataFrame([paper, ours])
display(comparison_df)

comparison_df.to_csv(
    OUT_DIR / "paper_vs_retrained_bugfixed_default_binary.csv",
    index=False
)

print(
    f"ACC difference vs paper: "
    f"{(ours['ACC'] - paper['ACC']) * 100:+.2f} percentage points"
)
print(
    f"MCC difference vs paper: "
    f"{ours['MCC'] - paper['MCC']:+.4f}"
)
print(
    f"AUROC difference vs paper: "
    f"{ours['AUROC'] - paper['AUROC']:+.4f}"
)
print(
    f"AUPR difference vs paper: "
    f"{ours['AUPR'] - paper['AUPR']:+.4f}"
)


# Added branch: CNN + Transformer ensemble

在原 DLBWE-Cys BiLSTM-Attention 模型基础上增加独立 CNN+Transformer 模型。

固定随机种子：
1. 数据集五折切分：
`StratifiedGroupKFold(random_state=SEED)`

2. Torch训练：
每次训练前：
`seed_everything(SEED)`

两个模型均使用同一个 SEED=1949，保证可重复。
最终采用 OOF MCC 最大的权重进行概率集成。

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=64):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )
        pe[:,0::2] = torch.sin(pos * div)
        pe[:,1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self,x):
        return x + self.pe[:,:x.size(1)]

class CNNTransformer(nn.Module):
    def __init__(self):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv1d(20,64,4),
            nn.ReLU(),
            nn.Conv1d(64,128,4),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        self.proj = nn.Linear(128,64)

        self.pos = PositionalEncoding(
            d_model=64,
            max_len=64
        )

        layer = nn.TransformerEncoderLayer(
            d_model=64,
            nhead=4,
            dim_feedforward=128,
            dropout=0.1,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            layer,
            num_layers=1
        )

        self.fc = nn.Sequential(
            nn.Linear(64,32),
            nn.ReLU(),
            nn.Linear(32,2)
        )

    def forward(self,x):

        x=x.permute(0,2,1)
        x=self.cnn(x)
        x=x.permute(0,2,1)
        x=self.proj(x)
        x=self.pos(x)
        x=self.encoder(x)
        x=x.mean(dim=1)
        return self.fc(x)


In [ ]:
def train_transformer_fold(fold_id, tr_idx, va_idx):

    seed_everything(TORCH_TRAIN_SEED)

    model=CNNTransformer().to(DEVICE)

    optimizer=torch.optim.Adam(
        model.parameters(),
        lr=LR
    )

    criterion=nn.CrossEntropyLoss()

    ds=TensorDataset(
        torch.from_numpy(X_train[tr_idx]).float(),
        torch.from_numpy(y_train[tr_idx]).long()
    )

    generator = torch.Generator()
    generator.manual_seed(TORCH_TRAIN_SEED)

    loader=DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator
    )

    best=-999
    best_epoch=0
    best_state=None
    best_prob=None

    for epoch in range(1,MAX_EPOCHS+1):

        model.train()

        for xb,yb in loader:
            xb=xb.to(DEVICE)
            yb=yb.to(DEVICE)

            optimizer.zero_grad()

            out=model(xb)
            loss=criterion(out,yb)

            loss.backward()
            optimizer.step()

        prob=predict_probabilities(
            model,
            X_train[va_idx],
            batch_size=BATCH_SIZE
        )

        m=binary_metrics(
            y_train[va_idx],
            prob,
            threshold=0.5
        )

        if m["MCC"]>best:
            best=m["MCC"]
            best_epoch=epoch
            best_state=copy.deepcopy(
                model.state_dict()
            )
            best_prob=prob.copy()

    model.load_state_dict(best_state)

    return model,best_epoch,best_prob


In [ ]:
sgkf_transformer = StratifiedGroupKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=CV_SPLIT_SEED
)

oof_transformer=np.zeros(len(y_train))
transformer_best_epochs=[]

for fold_id,(tr_idx,va_idx) in enumerate(
    sgkf_transformer.split(
        X_train,
        y_train,
        groups
    ),
    start=1
):

    print("Transformer fold:",fold_id)

    _,best_epoch_t,prob=train_transformer_fold(
        fold_id,
        tr_idx,
        va_idx
    )

    transformer_best_epochs.append(best_epoch_t)
    oof_transformer[va_idx]=prob

print(
    binary_metrics(
        y_train,
        oof_transformer
    )
)
print("Transformer best epochs:", transformer_best_epochs)


In [ ]:
best_w = FIXED_TRANSFORMER_WEIGHT

print(
    "Fixed Transformer weight:",
    best_w
)

print(
    "Fixed BiLSTM weight:",
    1.0 - best_w
)

ensemble_oof = (
    best_w * oof_transformer
    + (1.0 - best_w) * oof_prob
)

oof_compare_df = pd.DataFrame([
    {
        "Model": "CNN+BiLSTM-Attention",
        **binary_metrics(
            y_train,
            oof_prob,
            threshold=CV_THRESHOLD
        )
    },
    {
        "Model": "CNN+Transformer",
        **binary_metrics(
            y_train,
            oof_transformer,
            threshold=CV_THRESHOLD
        )
    },
    {
        "Model": "Fixed-weight Ensemble",
        **binary_metrics(
            y_train,
            ensemble_oof,
            threshold=CV_THRESHOLD
        )
    }
])

display(oof_compare_df)

oof_compare_df.to_csv(
    OUT_DIR / "OOF_fixed_weight_comparison.csv",
    index=False
)


# 原文结果对比：训练（5-fold CV）+ 独立测试

本部分同时报告并比较：

- **CNN+BiLSTM-Attention 单分支**
- **CNN+Transformer 单分支**
- **两模型概率集成**
- **原文 DLBWE-Cys**

训练阶段使用 5-fold cross-validation；测试阶段使用 `independent_set_test.fasta`。
集成权重仍然只由训练集 OOF 预测确定，独立测试集不用于选择权重。

In [ ]:
PAPER_CV = {
    "ACC": 0.7556,
    "SN": 0.6563,
    "SP": 0.8548,
    "MCC": 0.5218,
    "AUROC": 0.7998,
    "AUPR": 0.8244,
}

cv_compare_splitter = StratifiedGroupKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=CV_SPLIT_SEED
)

transformer_cv_rows = []
ensemble_cv_rows = []

for fold_id, (_, va_idx) in enumerate(
    cv_compare_splitter.split(
        X_train,
        y_train,
        groups=groups
    ),
    start=1
):
    mt = binary_metrics(
        y_train[va_idx],
        oof_transformer[va_idx],
        threshold=CV_THRESHOLD
    )

    me = binary_metrics(
        y_train[va_idx],
        ensemble_oof[va_idx],
        threshold=CV_THRESHOLD
    )

    transformer_cv_rows.append({
        "fold": fold_id,
        **mt
    })

    ensemble_cv_rows.append({
        "fold": fold_id,
        **me
    })

transformer_cv_df = pd.DataFrame(transformer_cv_rows)
ensemble_cv_df = pd.DataFrame(ensemble_cv_rows)

transformer_cv_df.to_csv(
    OUT_DIR / "CNN_Transformer_five_fold_results.csv",
    index=False
)

ensemble_cv_df.to_csv(
    OUT_DIR / "Ensemble_five_fold_results.csv",
    index=False
)

def cv_mean_sd_row(name, df):
    row = {"Model": name}
    for metric in ["ACC", "SN", "SP", "MCC", "AUROC", "AUPR"]:
        row[metric] = df[metric].mean()
        row[f"{metric}_SD"] = df[metric].std(ddof=1)
    return row

baseline_cv_row = cv_mean_sd_row(
    "CNN+BiLSTM-Attention",
    fold_summary_df
)

transformer_cv_row = cv_mean_sd_row(
    "CNN+Transformer",
    transformer_cv_df
)

ensemble_cv_row = cv_mean_sd_row(
    "Fixed-weight Ensemble",
    ensemble_cv_df
)

paper_cv_row = {
    "Model": "Original DLBWE-Cys paper (Table 2)",
    "ACC": 0.7556, "ACC_SD": 0.0021,
    "SN": 0.6563, "SN_SD": 0.0149,
    "SP": 0.8548, "SP_SD": 0.0164,
    "MCC": 0.5218, "MCC_SD": 0.0064,
    "AUROC": 0.7998, "AUROC_SD": 0.0089,
    "AUPR": 0.8244, "AUPR_SD": 0.0082,
}

cv_comparison_df = pd.DataFrame([
    paper_cv_row,
    baseline_cv_row,
    transformer_cv_row,
    ensemble_cv_row
])

print("===== TRAINING: 5-FOLD CV COMPARISON =====")
display(cv_comparison_df)

cv_comparison_df.to_csv(
    OUT_DIR / "TRAIN_CV_paper_vs_single_branches_vs_ensemble.csv",
    index=False
)

cv_publication_rows = []

for _, row in cv_comparison_df.iterrows():
    out = {"Model": row["Model"]}

    for metric in ["ACC", "SN", "SP", "MCC", "AUROC", "AUPR"]:
        out[metric] = (
            f"{row[metric]:.4f} ± {row[f'{metric}_SD']:.4f}"
        )

    cv_publication_rows.append(out)

cv_publication_df = pd.DataFrame(cv_publication_rows)

print("\n===== TRAINING: PUBLICATION-STYLE MEAN ± SD =====")
display(cv_publication_df)

cv_publication_df.to_csv(
    OUT_DIR / "TRAIN_CV_publication_mean_SD.csv",
    index=False
)

cv_delta_rows = []

for row in [baseline_cv_row, transformer_cv_row, ensemble_cv_row]:
    cv_delta_rows.append({
        "Model": row["Model"],
        "Delta_ACC": row["ACC"] - PAPER_CV["ACC"],
        "Delta_SN": row["SN"] - PAPER_CV["SN"],
        "Delta_SP": row["SP"] - PAPER_CV["SP"],
        "Delta_MCC": row["MCC"] - PAPER_CV["MCC"],
        "Delta_AUROC": row["AUROC"] - PAPER_CV["AUROC"],
        "Delta_AUPR": row["AUPR"] - PAPER_CV["AUPR"],
    })

cv_delta_df = pd.DataFrame(cv_delta_rows)

print("\n===== TRAINING: DELTA VS ORIGINAL PAPER =====")
display(cv_delta_df)

cv_delta_df.to_csv(
    OUT_DIR / "TRAIN_CV_delta_vs_original_paper.csv",
    index=False
)


In [ ]:
TRANSFORMER_FINAL_EPOCHS = max(
    int(np.median(transformer_best_epochs)),
    1
)

print("Transformer best epochs:", transformer_best_epochs)
print("Transformer final epochs:", TRANSFORMER_FINAL_EPOCHS)

seed_everything(TORCH_TRAIN_SEED)

final_transformer = CNNTransformer().to(DEVICE)

transformer_criterion = nn.CrossEntropyLoss()

transformer_optimizer = torch.optim.Adam(
    final_transformer.parameters(),
    lr=LR
)

transformer_ds = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train).long()
)

transformer_generator = torch.Generator()
transformer_generator.manual_seed(TORCH_TRAIN_SEED)

transformer_dl = DataLoader(
    transformer_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=transformer_generator,
    drop_last=False
)

transformer_final_log = []

for epoch in range(1, TRANSFORMER_FINAL_EPOCHS + 1):

    final_transformer.train()

    loss_sum = 0.0
    seen = 0

    for xb, yb in transformer_dl:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        transformer_optimizer.zero_grad(set_to_none=True)

        logits = get_logits(
            final_transformer(xb)
        )

        loss = transformer_criterion(
            logits,
            yb
        )

        loss.backward()
        transformer_optimizer.step()

        loss_sum += loss.item() * len(yb)
        seen += len(yb)

    epoch_loss = loss_sum / seen

    train_prob_t = predict_probabilities(
        final_transformer,
        X_train,
        batch_size=BATCH_SIZE
    )

    train_metrics_t = binary_metrics(
        y_train,
        train_prob_t,
        threshold=DEFAULT_THRESHOLD
    )

    transformer_final_log.append({
        "epoch": epoch,
        "train_loss": epoch_loss,
        **train_metrics_t
    })

    print(
        f"[FINAL Transformer | Epoch {epoch:03d}/{TRANSFORMER_FINAL_EPOCHS:03d}] "
        f"loss={epoch_loss:.5f} | "
        f"ACC={train_metrics_t['ACC']:.4f} "
        f"SN={train_metrics_t['SN']:.4f} "
        f"SP={train_metrics_t['SP']:.4f} "
        f"MCC={train_metrics_t['MCC']:.4f} "
        f"AUROC={train_metrics_t['AUROC']:.4f} "
        f"AUPR={train_metrics_t['AUPR']:.4f}"
    )

pd.DataFrame(
    transformer_final_log
).to_csv(
    OUT_DIR / "CNN_Transformer_final_training_log.csv",
    index=False
)

torch.save(
    {
        "model_state": final_transformer.state_dict(),
        "model_name": "CNN+Transformer",
        "final_epochs": TRANSFORMER_FINAL_EPOCHS,
        "seed": TORCH_TRAIN_SEED,
        "threshold": 0.5
    },
    OUT_DIR / "CNN_Transformer_final.pt"
)


In [ ]:
test_prob_bilstm = test_prob

test_prob_transformer = predict_probabilities(
    final_transformer,
    X_test,
    batch_size=BATCH_SIZE
)

test_prob_ensemble = (
    best_w * test_prob_transformer
    + (1.0 - best_w) * test_prob_bilstm
)

test_bilstm_metrics = binary_metrics(
    y_test,
    test_prob_bilstm,
    threshold=DEFAULT_THRESHOLD
)

test_transformer_metrics = binary_metrics(
    y_test,
    test_prob_transformer,
    threshold=DEFAULT_THRESHOLD
)

test_ensemble_metrics = binary_metrics(
    y_test,
    test_prob_ensemble,
    threshold=DEFAULT_THRESHOLD
)

PAPER_TEST = {
    "Model": "Original DLBWE-Cys paper (Table S3)",
    "ACC": 0.8445,
    "SN": 0.7590,
    "SP": 0.8530,
    "MCC": 0.4377,
    "AUROC": 0.8712,
    "AUPR": 0.4909,
}

test_comparison_df = pd.DataFrame([
    PAPER_TEST,
    {
        "Model": "CNN+BiLSTM-Attention",
        "ACC": test_bilstm_metrics["ACC"],
        "SN": test_bilstm_metrics["SN"],
        "SP": test_bilstm_metrics["SP"],
        "MCC": test_bilstm_metrics["MCC"],
        "AUROC": test_bilstm_metrics["AUROC"],
        "AUPR": test_bilstm_metrics["AUPR"],
    },
    {
        "Model": "CNN+Transformer",
        "ACC": test_transformer_metrics["ACC"],
        "SN": test_transformer_metrics["SN"],
        "SP": test_transformer_metrics["SP"],
        "MCC": test_transformer_metrics["MCC"],
        "AUROC": test_transformer_metrics["AUROC"],
        "AUPR": test_transformer_metrics["AUPR"],
    },
    {
        "Model": "Fixed-weight Ensemble",
        "ACC": test_ensemble_metrics["ACC"],
        "SN": test_ensemble_metrics["SN"],
        "SP": test_ensemble_metrics["SP"],
        "MCC": test_ensemble_metrics["MCC"],
        "AUROC": test_ensemble_metrics["AUROC"],
        "AUPR": test_ensemble_metrics["AUPR"],
    }
])

print("===== INDEPENDENT TEST: PAPER VS SINGLE BRANCHES VS ENSEMBLE =====")
display(test_comparison_df)

test_comparison_df.to_csv(
    OUT_DIR / "TEST_paper_vs_single_branches_vs_ensemble.csv",
    index=False
)

test_delta_rows = []

for _, row in test_comparison_df.iloc[1:].iterrows():
    test_delta_rows.append({
        "Model": row["Model"],
        "Delta_ACC": row["ACC"] - PAPER_TEST["ACC"],
        "Delta_SN": row["SN"] - PAPER_TEST["SN"],
        "Delta_SP": row["SP"] - PAPER_TEST["SP"],
        "Delta_MCC": row["MCC"] - PAPER_TEST["MCC"],
        "Delta_AUROC": row["AUROC"] - PAPER_TEST["AUROC"],
        "Delta_AUPR": row["AUPR"] - PAPER_TEST["AUPR"],
    })

test_delta_df = pd.DataFrame(test_delta_rows)

print("\n===== INDEPENDENT TEST: DELTA VS ORIGINAL PAPER =====")
display(test_delta_df)

test_delta_df.to_csv(
    OUT_DIR / "TEST_delta_vs_original_paper.csv",
    index=False
)

test_all_predictions_df = pd.DataFrame({
    "header": test_headers,
    "sequence": test_seqs,
    "true_label": y_test,
    "BiLSTM_prob_positive": test_prob_bilstm,
    "Transformer_prob_positive": test_prob_transformer,
    "Ensemble_prob_positive": test_prob_ensemble,
    "Ensemble_pred_0.5": (
        test_prob_ensemble >= 0.5
    ).astype(int)
})

test_all_predictions_df.to_csv(
    OUT_DIR / "TEST_predictions_single_branches_and_ensemble.csv",
    index=False
)

print(
    f"\nFinal ensemble weights: "
    f"Transformer={best_w:.2f}, "
    f"BiLSTM={1.0-best_w:.2f}"
)


In [ ]:
fixed_parameter_df = pd.DataFrame([{
    "CV_split_seed": CV_SPLIT_SEED,
    "Torch_training_seed": TORCH_TRAIN_SEED,
    "Transformer_weight": FIXED_TRANSFORMER_WEIGHT,
    "BiLSTM_weight": FIXED_BILSTM_WEIGHT,
    "CV_threshold": CV_THRESHOLD,
    "Test_threshold": DEFAULT_THRESHOLD,
    "Batch_size": BATCH_SIZE,
    "Learning_rate": LR,
    "Max_epochs": MAX_EPOCHS,
    "Patience": PATIENCE,
    "N_folds": N_FOLDS
}])

display(fixed_parameter_df)

fixed_parameter_df.to_csv(
    OUT_DIR / "fixed_parameters_seed1254_T025_B075.csv",
    index=False
)


In [ ]:
import shutil

archive_path = shutil.make_archive(
    "/content/DLBWE_fixed_seed1254_T025_B075_outputs",
    "zip",
    root_dir=OUT_DIR
)

print("All outputs:", OUT_DIR)
print("ZIP archive:", archive_path)
print("\nFiles:")
for p in sorted(OUT_DIR.iterdir()):
    print(" -", p.name)


## 重要说明

这份代码满足“Colab 左侧只上传两个 FASTA 文件”的要求，但它与此前
**Provided checkpoint bug-fixed** 的含义有一个本质区别：

- **此前那一行**：作者 checkpoint 已经固定，只修改 BiLSTM hidden 的组合方式，并采用默认二分类决策。
- **这份 Notebook**：没有作者 checkpoint，因此必须从 `fold_train.fasta` 重新训练模型。

所以，重新训练后的独立测试指标应以 Notebook 实际运行结果为准，**不能预先承诺一定得到 87.84%**。
如果要精确复现此前特定 checkpoint 的结果，仍然必须提供作者的 `DLBWE-Cys.pth`。